
<h1 style="color:#333; font-weight:500; letter-spacing:-0.5px;">
  <span style="opacity:0.4; font-size:1.2em; margin-right:8px;">◇</span>
  Compiling codes from all DX603
  <span style="opacity:0.4; font-size:1.2em; margin-left:8px;">◇</span>
</h1>

<hr>



In [ ]:
####
# common imports
####

import numpy                as np
import pandas               as pd 

import matplotlib.pyplot    as plt
import seaborn              as sns


from IPython.display        import display, HTML, Markdown

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Data Exploration · HW 2


</div>


In [ ]:
#      ►►►►►    Synthetic data generator    ◄◄◄◄◄◄
#      Purpose: Create a dummy based on regression

from sklearn.datasets       import make_regression
(synthetic_X
 , synthetic_y
 , synthetic_coef 
 ) = make_regression(
        n_samples=20
        , n_features=1
        # , n_informative=1
        , noise= 20
        , bias=0.5
        , coef=True
        , random_state=42
        )

In [ ]:
#      ►►►►►    simple regression    ◄◄◄◄◄◄
#      Purpose: typical workflow, fitting the model, then predict the new Y, then eval by MSE, MAE, RMSE

from sklearn.linear_model       import LinearRegression
linear_model_test1 = LinearRegression()
linear_model_test1.fit(synthetic_X, synthetic_y)
show_the_intercept = linear_model_test1.intercept_
show_the_coeff = linear_model_test1.coef_[0]

new_y_from_modelTest1 = linear_model_test1.predict(synthetic_X)


### ■■■■■■■■■■■■■■
###      Metrics       
### ■■■■■■■■■■■■■■

from sklearn.metrics            import mean_squared_error,root_mean_squared_error, mean_absolute_error, r2_score
MSE_from_modelTest1 = mean_squared_error(synthetic_y, new_y_from_modelTest1)


In [ ]:
#      ►►►►►    splitting data     ◄◄◄◄◄◄
#      Purpose: typical workflow for training, testing & evaluation sets

from sklearn.datasets           import load_diabetes
data_diabetes = load_diabetes(as_frame=True)
df_diabetes = pd.concat([data_diabetes.data, data_diabetes.target.rename('DiseaseProgression')], axis=1)
df_diabetes_x = df_diabetes.drop(columns=["DiseaseProgression"]).columns.tolist()



from sklearn.model_selection    import train_test_split    
(diab_train_x
 , diab_test_x
 , diab_train_y
 , diab_test_y 
 )= train_test_split(
    df_diabetes[df_diabetes_x]
    , df_diabetes["DiseaseProgression"]
    , test_size=0.2
    , random_state=42
    )


In [ ]:
#      ►►►►►    quick review    ◄◄◄◄◄◄
#      Purpose: simple premade code for reviewing

df_diabetes.hist(
    figsize=(12,8)
    , layout=(3,4)
    ,grid=False
    ,edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
#      ►►►►►    custom function: multitest    ◄◄◄◄◄◄
#      Purpose: Creating a repeated testing, based on different split

def mutlitest_on_splitRatio(ratio = [0.1, 0.2, 0.3, 0.4, 0.5])-> pd.DataFrame:
    """
    run different ratio of train-test and print out MSE for each
    """

    ratiolist = ratio

    mse_result_train = []
    mse_result_test = []

    for eachratio in ratiolist:

        (diab_train_x
        , diab_test_x
        , diab_train_y
        , diab_test_y 
        )= train_test_split(
            df_diabetes[df_diabetes_x]
            , df_diabetes["DiseaseProgression"]
            , test_size=eachratio
            , shuffle= True
            , random_state=42
            )

        model = LinearRegression()

        model.fit(diab_train_x, diab_train_y)

        diab_y_pred_train = model.predict(diab_train_x)
        diab_y_pred_test = model.predict(diab_test_x)

        mse_result_train.append(round(mean_squared_error(diab_train_y, diab_y_pred_train),2))
        mse_result_test.append(round(mean_squared_error(diab_train_y, diab_y_pred_test),2))

    ziplist = zip(ratiolist, mse_result_train, mse_result_test)

    return pd.DataFrame(ziplist, columns=['ratio', 'train_mse', 'test_mse'])


ratio_splir_test = [0.1, 0.2, 0.3, 0.4, 0.5]
mutlitest_on_splitRatio(ratio_splir_test)


<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Polynomial Data · HW 3a


</div>

<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Feature Engineering · HW 3b


</div>

### --------------Load data, in case

In [ ]:
# #### Load the dataset
# url = "http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
# column_names = ['mpg','cylinders','displacement','horsepower','weight',
#                 'acceleration','model_year','origin','car_name']
# df_mpg = pd.read_csv(url, sep=r'\s+', names=column_names, na_values='?')

           ------------    

###     ---- Work

In [ ]:
#      ►►►►►    simple data cleaning    ◄◄◄◄◄◄
#      Purpose: 

### ■■■■■■■■■■■■■■
###      dropping data       
### ■■■■■■■■■■■■■■

df_mpg_cleaned = (
    df_mpg
    .drop(columns='car_name')
    .dropna(subset=['horsepower'])
)

### ■■■■■■■■■■■■■■
###      creating a ratio feature      
### ■■■■■■■■■■■■■■

df_mpg_cleaned2 = (
    df_mpg_cleaned
    .assign(horsepower_sqrt = np.sqrt(df_mpg_cleaned['horsepower']))
    .assign(weight_logT = np.log1p(df_mpg_cleaned['weight']))
    .drop(columns=['horsepower', 'weight'])
)



<br>
<details>
<summary>📖 methodology notes</summary>

<br>


<h5 style="font-family:monospace;
           font-weight:400;
           color:#000;
           background:#f0f0f0;">Square-root transform of horsepower</h5>      
      

The `horsepower` variable is right-skewed, with a long tail of high values. We apply a square-root transformation to reduce skewness and compress the scale of extreme values:

$$
    \text{SqrtHP} = \sqrt{\text{Horsepower}}
$$

This helps the model treat high-horsepower cars more smoothly without being dominated by a few large values.


<h5 style="font-family:monospace;
           font-weight:400;
           color:#000;
           background:#f0f0f0;"> > Log-transform of weight </h5>

Car weight ranges widely and is heavily skewed, as shown in the histogram. We add a log-transformed feature:

$$
    \text{LogWeight} = \log(1 + \text{Weight})
$$

This transformation preserves order but compresses extreme values, allowing the model to better capture the non-linear influence of weight on fuel efficiency. 

**Note:** $log(1 + x)$ (also known as `log1p(x)` in numpy) is a common transformation because it behaves well across both small and large values, and avoids the error associated with 0 values in the data. 



<h5 style="font-family:monospace;
           font-weight:400;
           color:#000;
           background:#f0f0f0;">  Interaction term for displacement and cylinders </h5>

We hypothesize that the effect of engine displacement on MPG may depend on the number of cylinders, since larger engines typically have more cylinders. To model this interaction, we include:

$$
    \text{DispXyl} = \text{Displacement} \times \text{Cylinders}
$$

This allows the model to capture joint effects that wouldn’t be possible with separate linear terms.



**Note:** As in most modeling workflows, and **add** (not replace) new transformed features to enrich the model. In future lessons, we will explore how to evaluate the usefulness of these features and eliminate any that may add noise or redundancy.


<p>------------</p>


<h5 style="font-family:monospace;
           font-weight:400;
           color:#000;
           background:#f0f0f0;"> How to compute reductions</h5>

Suppose the following values are observed for a given metric (e.g., MAE):

* Baseline MAE = 5.00
* Feature-engineered MAE = 4.00

1. **Absolute reduction**

$$
\Delta \text{MAE}
= \text{baseline\_MAE} - \text{new\_MAE} \\
= 5.00 - 4.00 \\
= 1.0000
$$

2. **Percent reduction**

$$
\%\text{Reduction}
= \frac{\text{baseline\_MAE} - \text{new\_MAE}}{\text{baseline\_MAE}} \\
= \frac{1.00}{5.00} \\
= 0.2000
$$

</details>

<br>

        ------


In [ ]:
#      ►►►►►    intermezzo - pandas works    ◄◄◄◄◄◄
#      Purpose: jus common pandas technique to do EDA


### ■■■■■■■■■■■■■■
###      melt       
### ■■■■■■■■■■■■■■
metrictable_q2combined_melt = (
    pd.melt(
        metrictable_q2combined
        , id_vars = ['Metric', 'variables']
        , value_vars = ['training', 'evaluation']
        , var_name = 'split_set'
        , value_name = 'value'
    )
    )


### ■■■■■■■■■■■■■■
###      pivot_table       
### ■■■■■■■■■■■■■■

metrictable_q2_changes = (
    metrictable_q2combined_melt
    .pivot_table(
        index = ['Metric' , 'split_set']
        , columns = 'variables'
        , values = 'value'
        , aggfunc = 'mean'
    )
    .assign(
        abs_change = lambda df: df['baseline'] - df['engineered']
        , pct_change = lambda df: ((df['baseline'] - df['engineered']) / df['baseline']) * 100
    )
    .reset_index()
)



<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Imputation & Other Engineering · HW 4a


</div>

In [ ]:
#      ►►►►►    custom fuction: missing count    ◄◄◄◄◄◄
#      Purpose: custom viewer for counting all missingness, from Prof Snyder

from IPython.display import Markdown, display, HTML


def show_null_counts_features(df):
    # Count the nulls and calculate the %
    count_nulls = df.isnull().sum()
    df_nulls = (df.isnull().mean() * 100).round(2)
    
    # Determine if the column is numeric or non-numeric
    feature_types = df.dtypes.apply(lambda x: 'Numeric' if np.issubdtype(x, np.number) else 'Categorical')
    
    # Filter out the columns with missing values and sort them in descending order
    missing_data = pd.DataFrame({
        'Feature': count_nulls[count_nulls > 0].index,
        '# Null Values': count_nulls[count_nulls > 0].values, 
        'Null %': df_nulls[df_nulls > 0].values,
        'Type': feature_types[count_nulls > 0].values
    }).sort_values(by='Null %', ascending=False)
    
    display(HTML((f""" 
                  <div style="background:#fff3cd; padding:14px; border-left:5px solid #e0a800; border-radius:5px;">
                  The dataset contains {len(df)} samples.
                  <br>

                  The dataset contains {len(df.columns)} features.
                  
                  <br>

                  The dataset contains {len(missing_data['Feature'])} features with missing values.
                  
                  <br>

                  The dataset contains {len(df[df.duplicated()])} duplicate rows.\n
                  """)))

    if (len(missing_data) == 0):
        print("There are no null values in the dataset!")
    else:
        # Print null value stats
        print('Feature Name    # Nulls      Null %    Type')
        print('------------    -------      ------    ----')
        for index, row in missing_data.iterrows():
            print(f"{row['Feature']:<15} {row['# Null Values']:<12} {row['Null %']:.2f}%   {row['Type']}")




In [ ]:
#      ►►►►►    encoding    ◄◄◄◄◄◄
#      Purpose: simple encoding

from sklearn.preprocessing      import OrdinalEncoder, OneHotEncoder 

# Initialize OrdinalEncoder
encod = OrdinalEncoder()
col_encoded = encod.fit_transform(df.filter(categorical_features))

df_encoded = pd.DataFrame(col_encoded, columns=(df.filter(categorical_features)).columns, index=x_df2.index)


# Convert back to DataFrame to retain column names 
df_imputed = pd.concat([x_df2_encoded, y_df2], axis=1)

In [ ]:
#      ►►►►►    Imputation    ◄◄◄◄◄◄
#      Purpose: some steps for imputations

from sklearn.impute             import SimpleImputer

categorical_features = df.select_dtypes(exclude=['number']).columns.tolist()
numeric_features = df.select_dtypes(include=['number']).columns.tolist()

##___mode
df[categorical_features] = (
    SimpleImputer(strategy='most_frequent')
    .fit_transform(df[categorical_features])
)

##___mean
df[numeric_features] = (
    SimpleImputer(strategy='mean')
    .fit_transform(df[numeric_features])
)



<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Cross Validation · HW 4b


</div>

In [ ]:
#      ►►►►►    using CV    ◄◄◄◄◄◄
#      Purpose: cross_val_score is the cv from sklearn for quick cross validation


from sklearn.model_selection    import  cross_val_score, RepeatedKFold, LeaveOneOut



scores_cv = cross_val_score(
    LinearRegression()
    , x_train
    , y_train 

    # , cv = 5
    # , cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
    # , cv = LeaveOneOut()

    , scoring = 'neg_root_mean_squared_error'
)

root_mean_score_abs = scores_cv.mean()*-1



### ■■■■■■■■■■■■■■
###      multiple test custom function       
### ■■■■■■■■■■■■■■

def get_cv_score()->pd.DataFrame:
    "output as a DataFrame"


    ##_____ set _____ ##:
    model = LinearRegression()
    cv5=5
    rfk = RepeatedKFold(n_splits=5, n_repeats=100, random_state=42)
    loo=LeaveOneOut()
    listofcv = [cv5, rfk, loo]

    results_test = []


    ##_____ repetition _____ ##: 
    for typetodo in listofcv:
        s = cross_val_score(
            model,
            x_test, y_test,
            cv=typetodo,
            scoring='neg_root_mean_squared_error'
        )
        results_test.append(s.mean()*-1)
    
    return pd.DataFrame({'cv':["cv5", "rfk","loo"], 'score_test': results_test})



        
result_train = pd.DataFrame({'cv':["cv5", "rfk","loo"], 'score_train':[rootmean_scores_abs, rootmean_scores_abs2, rootmean_scores_abs3]})
result_test = get_cv_score()
 

<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Feature Selection  · HW 5


</div>

In [ ]:
#      ►►►►►    forward/backward selection    ◄◄◄◄◄◄
#      Purpose: to show example of auto selection, and the custom fucntion made by prof Snyder

### ■■■■■■■■■■■■■■
###      premade       
### ■■■■■■■■■■■■■■

from sklearn.feature_selection          import SequentialFeatureSelector
from sklearn.linear_model               import RidgeCV


ridgecv_setting = RidgeCV(alphas=np.logspace(-6, 6, num=5)).fit(X, y)
feat_todo = 2
direction = "forward" # "backward"

resulting_selection = (
    SequentialFeatureSelector(
        ridgecv_setting  
        , n_features_to_select= feat_todo 
        , direction= direction
    )
    .fit(X, y)
)


### ■■■■■■■■■■■■■■
###      custom forward selection from lecture       
### ■■■■■■■■■■■■■■


from sklearn.model_selection            import train_test_split, cross_val_score,RepeatedKFold
from sklearn.linear_model               import LinearRegression
from sklearn.metrics                    import mean_squared_error, r2_score


scoring_to_use = 'neg_mean_squared_error' 
cv_todo = 5
tol = None  #  0.0 (no further improvement)  #  1e-4 (point of diminishing return)
feat_todo = None 
memorycore_left = -2 



def forward_feature_selection(
        X
        , y 
        , model 
        , scoring
        , cv 
        , tol 
        , max_features 
        , n_jobs 
        , verbose
)-> list:
    
    ##_____ set _____ ##: 
    selected_features = []                      ####____◄ empty list - store the order of selected col
    remaining_features = list(X.columns)        ####____◄ checklist for features todo 
    best_scores = []                            ####____◄ store the CV score after each addition test
    previous_scores = float('inf')              ####____◄ initialize prev score for improvement comparison

    best_feature_set = None                     ####____◄ best combo found so far
    best_score = float('inf')                   ####____◄ best CV score so far


    ##_____ start the workflow _____ ##: 
    
    while remaining_features:                   
        scores = {}                             ####____◄ store the CV score after each addition test

        for feature in remaining_features:
            current_features = selected_features + [feature]
            
            ##_____ compute CV for current set & do negate _____ ##: 
            cvscore = -(cross_val_score(
                model 
                , X[current_features]
                , y 
                , scoring = scoring
                , cv = cv 
                , n_jobs = n_jobs                 
            ).mean()
            )

            scores[feature] = cvscore 


        ##_____ select the min score _____ ##: 
        best_feature = min(scores, key = scores.get)
        current_score = scores[best_feature]


        ##_____ check if improvement is signifcant based on tolerance _____ ##: 
        if tol is not None and previous_scores - current_score < tol:
            if verbose:
                print("Stopping early due to minimal improvement.")
            break


        ##_____ add the best feature to selected list _____ ##: 
        selected_features.append(best_feature)
        best_scores.append(current_score)
        remaining_features.remove(best_feature)
        previous_scores = current_score

        if verbose:
            print(f"\nFeatures: {selected_features[-3:]}, CV score ({scoring}): {current_score:.4f}")

        
        ##_____ update the best set _____ ##: 
        if current_score < best_score:
            best_score = current_score 
            best_feature_set = selected_features.copy()


        ##_____ max number check _____ ##: 
        if max_features is not None and len(selected_features) >= max_features:
            break


    return (
        selected_features
        , best_scores 
        , best_feature_set 
        , best_score
    )


### ■■■■■■■■■■■■■■
###      quick function for plot       
### ■■■■■■■■■■■■■■  

from matplotlib.pyplot                  import plot as plt 

def feature_selection_chart(
    selected_feature_varname = sel_feat
    , score_list_varname = scores    
    , best_set_varname = best_set 
    , best_score_varname = best_score
    , titletext
):
    
    ##_____ detecting the location of best set _____ ##: 
    index_feature = (
        np.argmax(
            np.array(selected_feature_varname) == selected_feature_varname[-1]
        )
    )


    plt.figure(figsize = (15, 3))
    
    plt.plog(range(1, len(score_list_varname) + 1), score_list_varname , marker ='.')
    plt.plot([index_feature +1], score_list_varname, marker = 'x' , color = 'red')

    plt.xticks(range(1, len(selected_feature_varname) + 1), selected_feature_varname
               , rotation = 45 
               , ha = 'right')
    
    plt.xlabel('Features added')
    plt.ylabel('CV score (RMSE)')

    plt.title(titletext)
    plt.annotate(f"Total features = {len(best_set_varname)}"
             , xy=(best_set_varname
                   .index(best_set_varname[-1])
                   , score_list_varname[2]
                   )
             , xytext=(best_set_varname
                       .index(best_set_varname[-1])
                       , score_list_varname[2]
            ), fontsize=10)
    
    plt.grid()
    plt.tight_layout()
    return plt.show()

    


<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Decision Tree · HW 6


</div>

In [ ]:
#      ►►►►►    do CV for decision tree     ◄◄◄◄◄◄
#      Purpose: custtom function, to do the tree hparam, do cross val, and do graph in one go

%matplotlib inline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RepeatedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.tree            import DecisionTreeRegressor, plot_tree
from sklearn.metrics         import mean_absolute_error


def run_decision_tree_regressor(
        X_train,
        y_train,
        criterion                = 'absolute_error'  ,  # Default parameters for DecisionTreeRegressor
        splitter                 = 'best',
        max_depth                = None,
        min_samples_split        = 2,
        min_samples_leaf         = 1,
        min_weight_fraction_leaf = 0.0,
        max_features             = None,
        random_state             = random_seed,         # Not the default, but we'll use it consistently for reproducibility
        max_leaf_nodes           = None,
        min_impurity_decrease    = 0.0,
        ccp_alpha                = 0.0,     
        n_jobs                   = -1,                  # Note: n_jobs affects cross-validation parallelism, not the tree itself.   
        visualize                = False,
        cv_n_splits             = 5,
        cv_n_repeats            = 5

        ): 
    
    """  
    """
    
    # Initialize the DecisionTreeRegressor
    decision_tree_model = DecisionTreeRegressor(criterion                = criterion,
                                                splitter                 = splitter,
                                                max_depth                = max_depth,
                                                min_samples_split        = min_samples_split,
                                                min_samples_leaf         = min_samples_leaf,
                                                min_weight_fraction_leaf = min_weight_fraction_leaf,
                                                max_features             = max_features,
                                                random_state             = random_state,  ###▔▔▔edited
                                                max_leaf_nodes           = max_leaf_nodes,
                                                min_impurity_decrease    = min_impurity_decrease,
                                                ccp_alpha                = ccp_alpha
                                               )

    
    # Perform cross-validation and return mean CV MAE
    neg_MAE_scores = cross_val_score(decision_tree_model, 
                                     X_train, 
                                     y_train,
                                     scoring='neg_mean_absolute_error',            
                                     cv=RepeatedKFold(
                                         n_splits= cv_n_splits,
                                         n_repeats= cv_n_repeats,
                                         random_state=random_seed
                                     ),

                                    #  random_state = random_state,    ###▔▔▔edited
                                     n_jobs=n_jobs)

    mean_cv_MAE = np.mean(abs(neg_MAE_scores))  # Convert negative MAE back to positive
    std_cv_MAE = np.std(abs(neg_MAE_scores))

    # Train the model on the full training set for visualization purposes
    if visualize:
        decision_tree_model.fit(X_train, y_train)  # Train on full training data for visualization
        plt.figure(figsize=(12, 6))
        plot_tree(
            decision_tree_model
            , feature_names= X_train.columns       ##feature_names   ###▔▔▔edited
            , filled=True
            , rounded=True
            , precision=4
            )
        plt.title(f"Decision Tree Structure (max_depth={max_depth})")
        plt.show()

    return mean_cv_MAE, std_cv_MAE


In [ ]:
#      ►►►►►    workflow 1: looping through 1 parameter    ◄◄◄◄◄◄
#      Purpose: This is a loop for a range of parameter for one feature only, the point is to do manual review on one feature

import time
from tqdm import tqdm

##_____ simple time recording _____ ##: 

start = time.time()


def format_time(seconds):
    
    # Convert seconds to hours, minutes, and remaining seconds
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = seconds % 60
    
    # Return a formatted string    
    if hours == 0 and minutes == 0:
        return f"{seconds:.2f}s"
    elif hours == 0:
        return f"{minutes}m {remaining_seconds:.2f}s"

    return f"{hours}h {minutes}m {remaining_seconds:.2f}s"




##_____ set up _____ ##: 

feature_to_do = (                                   ####____◄ feature to do manual test
    'max_depth'                                     ####____◄ select 1 only
    # 'min_samples_split'
    # 'min_samples_leaf'
    # 'min_weight_fraction_leaf'
    # 'max_features'
    # 'max_leaf_nodes'
    # 'min_impurity_decrease'
    # 'ccp_alpha'
)

hyperparameter_to_do = (                            ####____◄ depending on the feature, the range or list to do is different
    range(30, 51, 1)
)

list_MAE_result = []                                ####____◄ create a blank list so function will dump the results there


##_____ Do the loop _____ ##: 

for rangeparameter in tqdm(hyperparameter_to_do):
    list_MAE_result.append( 
        run_decision_tree_regressor(
            X_train                 = x_train
            , y_train               = y_train
            , criterion             = 'absolute_error'   
            , splitter              = 'best' 
            
            , max_depth             = rangeparameter  
            , min_samples_leaf      = '___' 
            , min_samples_split     = '___'  
            , max_features          = '___'
            , max_leaf_nodes        = '___' 
            
            , min_weight_fraction_leaf= '___'   
            , random_state          = random_seed
            , min_impurity_decrease = '___' 
            , ccp_alpha             = '___' 
            , n_jobs                = -2 
            , visualize             = True
        )
    )

end = time.time()
print(f"Execution Time of the loop: " + format_time(end-start))    

In [ ]:
#      ►►►►►    Workflow 1 - show graph    ◄◄◄◄◄◄
#      Purpose: This is graph for visualizing the results from workflow 1

import matplotlib.ticker            as mticker  

def dollar_format(x, pos):
    return f'${x:,.0f}'



##_____ helper info _____ ##: 
min_MAE = min(list_MAE_result)
min_MAE_index_loc = list_MAE_result.index(min_MAE)

if isinstance(hyperparameter_to_do[min_MAE_index_loc], int):
    print(f"Minimum MAE {min_MAE_index_loc:.4f} found at x = {hyperparameter_to_do[min_MAE_index_loc]}")
else:
    print(f"Minimum MAE {min_MAE_index_loc:.4f} found at x = {hyperparameter_to_do[min_MAE_index_loc]:.4f}")


##_____ show graph _____ ##: 
plt.figure(figsize=(6, 4))
plt.title("Decision Tree Regressor: MAE vs "+feature_to_do)
plt.plot(hyperparameter_to_do, list_MAE_result, marker='.')
plt.scatter([hyperparameter_to_do[min_MAE_index_loc]],[min_MAE_index_loc],marker='x',color='red')
plt.xlabel(feature_to_do)
plt.ylabel("MAE")
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
plt.grid()
plt.show()

In [ ]:
#      ►►►►►    Workflow 2: using GridSearch    ◄◄◄◄◄◄
#      Purpose: on top of the workflow 1, this one is using Grid Search to run for multiple features at the same time


import time
from tqdm import tqdm

##_____ simple time recording _____ ##: 

start = time.time()


def format_time(seconds):
    
    # Convert seconds to hours, minutes, and remaining seconds
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = seconds % 60
    
    # Return a formatted string    
    if hours == 0 and minutes == 0:
        return f"{seconds:.2f}s"
    elif hours == 0:
        return f"{minutes}m {remaining_seconds:.2f}s"

    return f"{hours}h {minutes}m {remaining_seconds:.2f}s"




##_____ set up _____ ##: 

features_for_multisearch =  {                        ####____◄ a set of features to be included, this is a dict now
    'max_depth'             : range(6,9, 1) 
    , 'max_leaf_nodes'      : range(90,94,1)  
    , 'max_features'        : range(55,66,3)
    , 'min_samples_split'   : range(30,36,2)
}


##_____ Do the Grid Search _____ ##: Grid model doesnt need the custom function above 

grid_model_setup = (
    GridSearchCV(
        estimator       = DecisionTreeRegressor(
                                criterion               = 'absolute_error' 
                                , splitter              = 'best'  
                                , min_samples_leaf      = 1  
                                , min_weight_fraction_leaf= 0.0 
                                , random_state          = 42 
                                , min_impurity_decrease = 0.0 
                                , ccp_alpha             = 0.0
                    )

        , param_grid    = {
                                'max_depth'             : range(6,9, 1) 
                                , 'max_leaf_nodes'      : range(90,94,1)  
                                , 'max_features'        : range(55,66,3)
                                , 'min_samples_split'   : range(30,36,2)
                            } 

        , cv            = RepeatedKFold(
                                n_splits            = 5
                                , n_repeats         = 5
                                , random_stat       = 42
                )                                     

        , n_jobs    =-2
        , scoring   ='neg_mean_absolute_error'  
        , verbose   =2
        )
)


##_____ fit the search to y train _____ ##: 

grid_model_setup.fit(x_train, y_train) 


##_____ keep in nicer setup _____ ##: 

grid_result_df = (
    pd.DataFrame(
        grid_model_setup
    ) [
        [
        'param_max_depth'
        , 'param_max_leaf_nodes'
        , 'param_max_features'  
        , 'param_min_samples_split'
        , 'mean_test_score'
        ]
      ]
)

end = time.time()
print(f"Execution Time: " + format_time(end-start))

<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Gradient Boosting · HW 7

</div>

In [ ]:
#      ►►►►►    basic run    ◄◄◄◄◄◄
#      Purpose: just the typical model

from sklearn.ensemble        import GradientBoostingRegressor

gradient_model = GradientBoostingRegressor(
    n_estimators        =000   
    , max_depth         =000
    , max_features      =000
    , min_samples_split =000
    , min_samples_leaf  =000
    , random_state      =42
)

gradient_model.fit(x_train, y_train)
y_pred_from_gradient_model = gradient_model.predict(x_test)
MAE_from_gradient_model = mean_absolute_error(y_test, y_pred_from_gradient_model)

In [ ]:
#      ►►►►►    Workflow 1: custom function - sweeping    ◄◄◄◄◄◄
#      Purpose: From HW, this is just a function to do repetitive manual work for one feature at a time

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble        import GradientBoostingRegressor
from sklearn.metrics         import mean_absolute_error
from tqdm                    import tqdm
import time


### ✸✸✸✸✸✸✸✸✸✸✸
###             prepare funcitons
### ✸✸✸✸✸✸✸✸✸✸✸

def run_model(model, 
              X_train, y_train, 
              n_repeats=5, 
              n_jobs=-1, 
              **model_params
             ):

    # Instantiate the model if a class is provided
    if isinstance(model, type):
        model = model(**model_params)
    else:                                    
        model.set_params(**model_params)    

    # Use negative MAE for cross-validation (since sklearn minimizes loss)
    neg_mae_scores = cross_val_score(
        model, 
        X_train, y_train,
        scoring='neg_mean_absolute_error',
        cv=RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=random_seed), 
        n_jobs=n_jobs
    )
    
    mean_cv_mae = -np.mean(neg_mae_scores)
    std_cv_mae  =  np.std(neg_mae_scores)
    
    # Fit the model on the full training set
    model.fit(X_train, y_train)
    
    # Compute training MAE
    train_preds = model.predict(X_train)
    train_mae   = mean_absolute_error(y_train, train_preds)
    
    return mean_cv_mae, std_cv_mae, train_mae

def sweep_parameter(model,
                    Parameters,
                    param,
                    parameter_list,
                    X_train          = X_train,              # Defined above
                    y_train          = y_train,
                    verbose          = True,
                    show_mae         = True,
                    show_std         = False,
                    n_iter_no_change = None,
                    delta            = 0.001,
                    n_jobs           = -1,
                    n_repeats        = 5
                   ):
    

    start = time.time()
    Parameters = Parameters.copy()  # Avoid modifying the original dictionary
    
    cv_maes, std_cvs, train_maes = [], [], []
    no_improve_count = 0
    best_mae = float('inf')
    
    # Run over each value in parameter_list
    for p in tqdm(parameter_list, desc=f"Sweeping {param}"):
        Parameters[param] = p
        P_temp = Parameters.copy()
        P_temp.pop('MAE_found', None)  # Just in case
        
        cv_mae, std_cv, train_mae = run_model(
            model=model,
            X_train=X_train, y_train=y_train,
            n_repeats=n_repeats,
            n_jobs=n_jobs,
            **P_temp
        )
        cv_maes.append(cv_mae)
        std_cvs.append(std_cv)
        train_maes.append(train_mae)
        
        if cv_mae < best_mae - delta:
            best_mae = cv_mae
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if n_iter_no_change is not None and no_improve_count >= n_iter_no_change:
            print(f"Early stopping: No improvement after {n_iter_no_change} iterations.")
            break

    # Identify best parameter
    min_cv_mae = min(cv_maes)
    min_index = cv_maes.index(min_cv_mae)
    best_param = parameter_list[min_index]
    Parameters[param] = best_param
    Parameters['MAE_found'] = min_cv_mae

    # ---------- Plotting section ----------
    if verbose:
        partial_param_list = parameter_list[:len(cv_maes)]

        is_boolean = all(isinstance(val, bool) for val in partial_param_list)
        if is_boolean:
            x_vals = list(range(len(partial_param_list)))
            x_labels = [str(val) for val in partial_param_list]
        else:
            x_vals = partial_param_list
            x_labels = partial_param_list

        error_name = 'MAE'

        # Create appropriate number of subplots
        if show_std:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
        else:
            fig, ax1 = plt.subplots(1, 1, figsize=(8, 4))

        ax1.set_title(f"{error_name} vs {param}")
        if show_mae:
            ax1.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))

        ax1.plot(x_vals,
                 cv_maes,
                 marker='.', label="CV MAE", color='blue')
        ax1.plot(x_vals,
                 train_maes,
                 marker='.', label="Train MAE", color='green')
        ax1.scatter([x_vals[min_index]],
                    [min_cv_mae],
                    marker='x', label="Best CV MAE", color='red')

        ax1.set_ylabel(error_name)
        ax1.legend()
        ax1.grid()

        # Optional Std Dev Plot
        if show_std:
            ax2.set_title(f"CV Standard Deviation vs {param}")
            ax2.plot(x_vals, std_cvs, marker='.', label="CV MAE Std", color='blue')
            ax2.set_xlabel(param)
            ax2.set_ylabel("Standard Deviation")
            ax2.legend()
            ax2.grid(alpha=0.5)

            if is_boolean:
                ax2.set_xticks(x_vals)
                ax2.set_xticklabels(x_labels)
        else:
            ax1.set_xlabel(param)
            if is_boolean:
                ax1.set_xticks(x_vals)
                ax1.set_xticklabels(x_labels)

        plt.tight_layout()
        plt.show()

        end = time.time()
        print("Execution Time:", time.strftime("%H:%M:%S", time.gmtime(end - start)))

    return Parameters


### ✸✸✸✸✸✸✸✸✸✸✸
###             run
### ✸✸✸✸✸✸✸✸✸✸✸

Default_Parameters_GB = {
    'n_estimators'            : 100,             # The number of boosting stages to be run. More estimators can improve performance but increase training time.
    'max_depth'               : 3,               # Maximum depth of individual trees. Controls model complexity.
    'max_features'            : None,            # Number of features to consider when looking for best split. Can help reduce overfitting.
    'min_samples_split'       : 2,               # Defines the minimum number of samples required to split an internal node.
    'min_samples_leaf'        : 1,               # Specifies the minimum number of samples that must be present in a leaf node. 
    'random_state'            : random_seed,     # Controls randomness of boosting. Useful for reproducibility.
    'MAE_found'               : float('inf')     # NOT a model parameter, but will record the MAE found for the current parameter choices
}

Params_GB = Default_Parameters_GB.copy()

compileresult = []
parametertest = 'n_estimators'

test_subset = range(10,1211, 300)
# test_subset = range(Params_GB[parametertest] - 100, Params_GB[parametertest]+ 100, 50)



Params_GB = sweep_parameter(
    model=GradientBoostingRegressor() 
    , Parameters=Params_GB 
    , param                 = parametertest
    , parameter_list        = test_subset
    , verbose=True
    , n_repeats=1
)


compileresult.append(Params_GB)



In [ ]:
#      ►►►►►    Workflow 2: automate using RandomizedSearch    ◄◄◄◄◄◄
#      Purpose: Now, this is using premade packages

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble        import GradientBoostingRegressor
from sklearn.metrics         import mean_absolute_error
from tqdm                    import tqdm
from scipy.stats             import uniform, randint

import time


param_for_ransearch = {
    'n_estimators': randint(870,896)
    , 'max_depth': randint(1,6)
    , 'max_features': randint(35,46)
    , 'min_samples_split': randint(2,5)
    , 'min_samples_leaf': randint(1,4)
}



n_iter_todo = 60
n_repeats_todo  = 5


randomsearch = RandomizedSearchCV(
    estimator = GradientBoostingRegressor()
    , param_distributions = param_for_ransearch
    , n_iter = n_iter_todo
    , n_jobs = -2
    , cv = RepeatedKFold(n_splits=5, n_repeats=n_repeats_todo, random_state=42)
    , scoring='neg_mean_absolute_error'
    , verbose = 1
    , random_state = 42

)

randomsearch.fit(X_train, y_train)

print(randomsearch.best_params_)

<hr>

<div style="font-family:system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI';
           font-weight:600;
           letter-spacing:0.08em;
           text-transform:uppercase;
           color:#333;
           border-bottom:2px solid #ccc;
           padding-bottom:4px;">

           
  # ◼ Using Classification · HW 8


</div>